In this code I attemp to perform the ACF procedure for production estimation

In [77]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using Random
using Distributions
using Optim
using Plots
using ShiftedArrays  # for lag function

In [78]:
# [NOTE TO SELF]: save the value-added calculation for later and just to the raw dataset for now

# Step 0: preparing the dataset - calculating value-addded production function 
dataset = CSV.read("op_lp_ready.csv", DataFrame)

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64
1,1,1997,15.2437,12.2967,13.6469,14.7629,27.4075
2,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425
3,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536
4,2,1997,14.8474,12.3572,13.6883,14.2698,43.5369
5,2,1998,14.7601,11.5891,13.6968,14.2355,41.273
6,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179
7,3,1998,13.9124,12.1451,12.0143,13.4115,17.4865
8,3,1999,13.66,9.89273,12.0052,13.0285,17.4949
9,4,1995,17.6699,14.5626,17.1751,17.2448,71.6182


Now we proceed with step 1 for ACF 

Instead of a linear labor term, I will fit a second-order polynomials for all our variables (labor, capital, materials).

I chose to fit a full second-order polynomial with interactions 

In [79]:
# A0: generating the polynomial terms 
dataset.v_capital_square = dataset.v_capital .* dataset.v_capital # now square terms
dataset.v_material_square = dataset.v_material .* dataset.v_material
dataset.v_labor_square = dataset.v_labor .* dataset.v_labor
dataset.int_capital_material = dataset.v_capital .* dataset.v_material # now interaction terms 
dataset.int_capital_labor = dataset.v_capital .* dataset.v_labor
dataset.int_material_labor = dataset.v_material .* dataset.v_labor

# A1: running the step 1 regression 
step1 = lm(@formula(v_production ~ v_labor + v_capital + v_material +
                    v_labor_square + v_capital_square + v_material_square +
                    int_capital_material + int_capital_labor + int_material_labor), dataset)

display(step1)

# B0: calculating the necessary variables for step 2 
# calculating residualized production without labor 

# # calculating predicted phi 
dataset.predicted_phi = (coef(step1)[1] .+ 
                            dataset.v_labor .* coef(step1)[2] .+
                            dataset.v_capital .* coef(step1)[3] .+
                            dataset.v_material .* coef(step1)[4] .+ 
                            dataset.v_labor_square .* coef(step1)[5] .+
                            dataset.v_capital_square .* coef(step1)[6] .+ 
                            dataset.v_material_square .* coef(step1)[7] .+
                            dataset.int_capital_material .* coef(step1)[8] .+ 
                            dataset.int_capital_labor .* coef(step1)[9] .+
                            dataset.int_material_labor .* coef(step1)[10])

# # now we delete the old variables 
select!(dataset, Not([:v_material_square, :v_capital_square, :v_labor_square, :int_capital_material, :int_capital_labor, :int_material_labor]))

# # # B0: now we calculate the lagged phi needed for step 2: lag_phi and lag_capital 
sort!(dataset, [:firm_id, :year])
transform!(groupby(dataset, :firm_id), :v_capital => (x -> lag(x, 1)) => :lag_capital) #generate capital lag variable
transform!(groupby(dataset, :firm_id), :predicted_phi => (x -> lag(x, 1)) => :lag_phi) # generate phi lag variable
transform!(groupby(dataset, :firm_id), :v_labor => (x -> lag(x, 1)) => :lag_labor) # generate material lag labor

dataset = dropmissing(dataset, [:lag_capital, :lag_phi, :lag_labor])

display(first(dataset, 10))

StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

v_production ~ 1 + v_labor + v_capital + v_material + v_labor_square + v_capital_square + v_material_square + int_capital_material + int_capital_labor + int_material_labor

Coefficients:
────────────────────────────────────────────────────────────────────────────────────────────
                             Coef.   Std. Error       t  Pr(>|t|)     Lower 95%    Upper 95%
────────────────────────────────────────────────────────────────────────────────────────────
(Intercept)            7.7274       0.13414       57.61    <1e-99   7.46446       7.99034
v_labor                0.0560451    0.00173096    32.38    <1e-99   0.0526521     0.0594381
v_capital              0.332844     0.0213801     15.57    <1e-53   0.290935      0.374752
v_material            -0.350749     0.0235883    -14.87    <1e-48  -0.396986     -

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,predicted_phi,lag_capital,lag_phi,lag_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425,15.3359,13.6469,15.1288,27.4075
2,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536,15.3177,13.8412,15.3359,20.8425
3,2,1998,14.7601,11.5891,13.6968,14.2355,41.273,14.8187,13.6883,14.861,43.5369
4,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179,14.7626,13.6968,14.8187,41.273
5,3,1999,13.66,9.89273,12.0052,13.0285,17.4949,13.5147,12.0143,13.8017,17.4865
6,4,1996,17.6065,15.2652,17.2278,17.2083,78.801,17.7206,17.1751,17.7238,71.6182
7,5,1999,16.0031,14.0184,15.313,15.6114,51.9997,16.1248,15.1472,16.1403,52.9994
8,6,1999,11.0108,8.07091,8.89698,9.20659,-4.84692,10.4205,8.54161,10.3449,-5.17222
9,7,1997,12.5013,8.65869,9.51618,11.9876,16.5183,12.4935,9.18124,12.0783,14.1084


I define the exogeneity moment condition as the full residual (xi + epsilon) being exogenous to capital, lag_capital, labor, lag_labor, lag_phi, and 1


In [80]:
# B0: defining the crucial functions for our GMM procedure 

function markov(parameter_guess, lag_phi, lag_labor, lag_capital)
    beta_k = parameter_guess[1]
    beta_l = parameter_guess[2]
    alpha_0 = parameter_guess[3]
    alpha_1 = parameter_guess[4]

    # first-order markov process: now = alpha + alpha_1 * last
    value = alpha_0 .+ alpha_1 .* (lag_phi - beta_k .* lag_capital - beta_l .* lag_labor)

    return value
end

function objective_function(
    parameter_guess, 
    production, 
    lag_phi, 
    labor,
    capital, 
    lag_labor, 
    lag_capital
)
    beta_k = parameter_guess[1]
    beta_l = parameter_guess[2]
    alpha_0 = parameter_guess[3]
    alpha_1 = parameter_guess[4]    

    RHS = (beta_k .* capital) .+ (beta_l .* labor) .+ markov(parameter_guess, lag_phi, lag_labor, lag_capital)

    # this is the main difference compared to OP in my opinion 
    epsilon = production - RHS

    instrument = Matrix(hcat(ones(length(capital)),capital,lag_capital,labor,lag_labor, lag_phi))  # Nx6 matrix
    g = (transpose(epsilon) * instrument) ./ length(instrument)
    loss = dot(g, g)  # equivalent to g'T * g
    return loss
end 

objective_function (generic function with 1 method)

In [81]:
# # tester function 
display(first(dataset, 10))
objective_function([0,0,0,0], 
    dataset.v_production, 
    dataset.lag_phi, 
    dataset.v_labor, 
    dataset.v_capital, 
    dataset.lag_labor, 
    dataset.lag_capital
)

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,predicted_phi,lag_capital,lag_phi,lag_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425,15.3359,13.6469,15.1288,27.4075
2,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536,15.3177,13.8412,15.3359,20.8425
3,2,1998,14.7601,11.5891,13.6968,14.2355,41.273,14.8187,13.6883,14.861,43.5369
4,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179,14.7626,13.6968,14.8187,41.273
5,3,1999,13.66,9.89273,12.0052,13.0285,17.4949,13.5147,12.0143,13.8017,17.4865
6,4,1996,17.6065,15.2652,17.2278,17.2083,78.801,17.7206,17.1751,17.7238,71.6182
7,5,1999,16.0031,14.0184,15.313,15.6114,51.9997,16.1248,15.1472,16.1403,52.9994
8,6,1999,11.0108,8.07091,8.89698,9.20659,-4.84692,10.4205,8.54161,10.3449,-5.17222
9,7,1997,12.5013,8.65869,9.51618,11.9876,16.5183,12.4935,9.18124,12.0783,14.1084


8805.210203163138

In [82]:
# function to run the GMM procedure 
function GMM_main(
    initial_guess, 
    dataset
)
    result = optimize(parameter_guess -> objective_function(
            parameter_guess,
            dataset.v_production, 
            dataset.lag_phi, 
            dataset.v_labor, 
            dataset.v_capital, 
            dataset.lag_labor, 
            dataset.lag_capital
        ),
        initial_guess,
        NelderMead()
    )
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)

    parameter_estimated = Optim.minimizer(result)
    println(parameter_estimated)
    println("Estimated capital coefficient: ", parameter_estimated[1])
    println("Estimated labor coefficient: ", parameter_estimated[2])
end 

GMM_main([0.1,0.1,0.1,0.1], dataset)

println("Mean productivity values: ", mean(dataset.predicted_phi))

Minimum loss value: 3.4602196871031106e-5
[0.9777804914475001, 0.007477488041176629, 0.10740759405630815, 0.9198186041856862]
Estimated capital coefficient: 0.9777804914475001
Estimated labor coefficient: 0.007477488041176629
Mean productivity values: 13.83633890021996
